# NYC Taxi Dataset - Big Data e Computação em Nuvem
## Machine Learning & Pipelines

### Documentação: [PIPELINE SPARK](https://spark.apache.org/docs/latest/ml-pipeline.html)


* pickup_datetime: The date and time when the meter was engaged.
* dropoff_datetime: The date and time when the meter was disengaged
* passenger_count: The number of passengers in the vehicle. This is a driver-entered value
* fare_amount: The time-and-distance fare calculated by the meter

## Import de bibliotecas

In [1]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

import pyspark.sql.functions as f
from pyspark.sql.types import StringType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.feature import OneHotEncoder, StringIndexer
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

## Criação Spark Session

In [2]:
# Criar a sessao do Spark
from pyspark.sql import SparkSession
spark = SparkSession \
            .builder \
            .master("local[*]") \
            .appName("nyc_<mudar-nome>") \
            .config("spark.executor.memory", "4g") \
            .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/24 19:04:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark

## Leitura dos Dados

In [4]:
from pyspark.sql.types import *

labels = (('key', TimestampType()),
          ('fare_amount', FloatType()),
          ('pickup_datetime', TimestampType()),
          ('pickup_longitude', FloatType()),
          ('pickup_latitude', FloatType()),
          ('dropoff_longitude', FloatType()),
          ('dropoff_latitude', FloatType()),
          ('passenger_count', IntegerType()))
          

schema = StructType([StructField(x[0], x[1], True) for x in labels])

In [5]:
#StructType([StructField("f1", StringType(), True)])
#True significa se podem haver dados nulos nas colunas

In [6]:
df = spark.read.csv("/home/pads/notebooks/PADSONL07/dados/nyc-taxi/train.csv", header=True, schema=schema)

In [7]:
df.schema

StructType([StructField('key', TimestampType(), True), StructField('fare_amount', FloatType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('pickup_longitude', FloatType(), True), StructField('pickup_latitude', FloatType(), True), StructField('dropoff_longitude', FloatType(), True), StructField('dropoff_latitude', FloatType(), True), StructField('passenger_count', IntegerType(), True)])

## Feature Engineering

## Train/Test Split

* [pyspark.sql.DataFrame.randomSplit](https://spark.apache.org/docs/3.1.1/api/python/reference/api/pyspark.sql.DataFrame.randomSplit.html)
* [pyspark.sql.DataFrame.sample](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.sql.DataFrame.sample.html)

## Feature Engineering: One-Hot-Encoding

* [pyspark.ml.feature.StringIndexer](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StringIndexer.html)
* [pyspark.ml.feature.OneHotEncoder](https://spark.apache.org/docs/3.1.1/api/python/reference/api/pyspark.ml.feature.OneHotEncoder.html)

## Feature Engineering: Feature Normalization

* [pyspark.ml.feature.VectorAssembler](https://spark.apache.org/docs/3.1.1/api/python/reference/api/pyspark.ml.feature.VectorAssembler.html)
* [pyspark.ml.feature.StandardScaler](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.feature.StandardScaler.html)

## Assembling dos vetores

## Visualizando as transformações

## Criação do Pipeline

* [pyspark.ml.regression.LinearRegression](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.LinearRegression.html)
* [pyspark.ml.Pipeline](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.Pipeline.html)

1. **LinearRegression**: Esta é a classe usada para criar um modelo de regressão linear. A regressão linear é uma técnica estatística usada para modelar a relação entre uma variável dependente e uma ou mais variáveis independentes.

2. **maxIter=50**: Este parâmetro define o número máximo de iterações para o algoritmo de otimização. Aqui, está definido como 50, o que significa que o processo de otimização (por exemplo, a descida do gradiente) será executado no máximo 50 vezes.

3. **solver='normal'**: Define o método de solução a ser usado para calcular os coeficientes da regressão. No caso 'normal', é provável que se refira ao método dos mínimos quadrados normais.

4. **labelCol='fare_amount'**: Especifica o nome da coluna no conjunto de dados que contém a variável dependente (ou a variável que o modelo está tentando prever). Neste caso, é a coluna 'fare_amount', que pode representar, por exemplo, o valor da tarifa em um conjunto de dados de transporte.

5. **featuresCol='features_vector'**: Indica a coluna que contém as variáveis independentes (features) no formato de um vetor. 'features_vector' é provavelmente uma coluna que contém um vetor de características consolidadas a partir de várias colunas de entrada.

6. **elasticNetParam=0.2**: Este parâmetro é usado no contexto da regularização Elastic Net, que é uma combinação de L1 (Lasso) e L2 (Ridge) regularizações. O valor 0.2 indica que há uma mistura de 20% de Lasso e 80% de Ridge.

7. **regParam=0.02**: Este é o parâmetro de regularização. Regularização é uma técnica usada para evitar o sobreajuste (overfitting) do modelo aos dados de treinamento. Um valor de 0.02 é relativamente baixo, indicando uma pequena quantidade de regularização.

Em resumo, o código está configurando um modelo de regressão linear para prever a variável 'fare_amount' com base em características fornecidas na coluna 'features_vector', usando um método de solução normal com um máximo de 50 iterações e aplicando uma regularização leve via Elastic Net.

## Model Training

## Model performance evaluation

* [pyspark.ml.evaluation.RegressionEvaluator](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html)

## Hyperparameter Tuning

* [pyspark.ml.tuning.ParamGridBuilder](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.ParamGridBuilder.html)
* [pyspark.ml.tuning.CrossValidator](https://spark.apache.org/docs/3.1.1/api/python/reference/api/pyspark.ml.tuning.CrossValidator.html)

In [25]:
en = [0.2, 0.3]
reg = [0.02, 0.03]
elastic_net = [e for e in en for r in reg]
regularization = [r for e in en for r in reg]

rmse_df = pd.DataFrame({'rmse':cv_model.avgMetrics,
                        'elastic_net_alpha': elastic_net, 
                        'regularization_term': regularization})

rmse_df.sort_values(by='rmse')

,rmse,elastic_net_alpha,regularization_term
0,9.775827,0.2,0.02
2,9.777949,0.3,0.02
1,9.778184,0.2,0.03
3,9.779669,0.3,0.03


In [26]:
elastic_net

[0.2, 0.2, 0.3, 0.3]

In [27]:
regularization

[0.02, 0.03, 0.02, 0.03]